# YieldMPNN 数据处理教程

## 概述

本 notebook 用于说明 YieldMPNN 的数据是如何从原始 reaction SMILES 一步步变成模型可消费的图结构的。

和原仓库相比，这里不会直接执行 `data/get_data.py` 然后得到一个黑盒 `.npz` 文件，而是把它拆成 4 个可观察的阶段：

1. 读取原始反应记录
2. 解析 reactants / products 并做最基本标准化
3. 把每个分子转成节点特征、边特征和边索引
4. 按仓库的 slot 规则压缩成 `dataset.py` 能恢复的结构

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 读取教学小样本与原始 split |
| 2 | 解析 reaction SMILES |
| 3 | 重现 `get_data.py` 的特征工程 |
| 4 | 单分子转图示例 |
| 5 | 多反应压缩为仓库风格数据结构 |
| 6 | 恢复为 DGL 图对象 |
| 7 | 与原仓库处理结果对照 |
| 8 | 总结与下一步 |

### 下面这个代码单元会做什么？

它会完成本 notebook 的运行时初始化：导入 `numpy / pandas / torch / dgl / rdkit`，定位项目根目录和外部源码仓库，并准备后面所有步骤公用的路径常量。
补充说明：这一格还会先设置 `MKL_THREADING_LAYER=GNU` 与 `OMP_NUM_THREADS=1`，避免后续图神经网络前向计算在某些 CPU-only PyTorch / MKL 组合下触发 OpenMP 线程层冲突。


In [ ]:
# ====== Step 0: 环境初始化 ======
from pathlib import Path
from contextlib import contextmanager
import os

# 先设置数值库线程策略，避免某些 CPU-only PyTorch / MKL 组合触发 OpenMP 符号问题。
os.environ.setdefault('MKL_THREADING_LAYER', 'GNU')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import sys
import numpy as np
import pandas as pd
import torch
import dgl

from rdkit import Chem, RDConfig
from rdkit.Chem import ChemicalFeatures


def locate_project_root(start: Path | None = None, root_name: str = 'Chemical_Synthesis') -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if candidate.name == root_name:
            return candidate
    raise RuntimeError(f'Cannot locate project root named {root_name!r} from {path}')


@contextmanager
def pushd(path: Path):
    old = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)


PROJECT_ROOT = locate_project_root()
TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos/5.reaction_yields_tutorial/YieldMPNN'
SOURCE_REPO = (PROJECT_ROOT / '../../../../../data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction_yield_nn').resolve()
DEMO_CSV = TUTORIAL_DIR / 'data/yieldmpnn_demo_reactions.csv'
RAW_SPLIT = SOURCE_REPO / 'data/split/data1_split_0.npz'
PROCESSED_DATA = SOURCE_REPO / 'data/dataset_1_0.npz'

chem_feature_factory = ChemicalFeatures.BuildFeatureFactory(os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef'))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SOURCE_REPO =', SOURCE_REPO)
print('DEMO_CSV =', DEMO_CSV)
print('RAW_SPLIT =', RAW_SPLIT)
print('PROCESSED_DATA =', PROCESSED_DATA)

## 1. 读取教学小样本与原始 split

### 为什么先从小样本开始？

原始 Buchwald-Hartwig split 一次包含数千条反应。直接对全量数据做特征工程，读者会很难看清“每一条记录到底发生了什么”。

因此这里先读取：

- 一个教学小样本 CSV
- 一个原始 `data1_split_0.npz`

这样既能看到真实数据分布，又能用很小的样本把完整流程讲清楚。

### 源码对应

- 文件：`data/get_data.py`
- 对应输入：`data/split/data1_split_*.npz`

### 下面这个代码单元会做什么？

它会同时读取教学小样本 CSV 和原始 `data1_split_0.npz`，让读者先建立“教学子集”和“真实全量 split”之间的对应关系。



In [ ]:
# ====== Step 1: 读取教学小样本与原始 split ======
demo_df = pd.read_csv(DEMO_CSV)
raw_split = np.load(RAW_SPLIT, allow_pickle=True)['data_df']
raw_df = pd.DataFrame(raw_split, columns=['reaction_smiles', 'yield'])
raw_df['yield'] = raw_df['yield'].astype(float)

print('教学小样本条数:', len(demo_df))
print('原始 split 条数:', len(raw_df))
print()
print('教学小样本预览:')
display(demo_df[['sample_id', 'row_index', 'yield']])
print('原始 split 前 3 条:')
display(raw_df.head(3))

## 2. 解析 reaction SMILES

### 为什么需要这一步？

原始记录是一个 reaction SMILES 字符串，例如：

`reactant_1.reactant_2....reactant_n >> product_1.product_2...`

但 YieldMPNN 并不是把整条字符串直接喂给模型，而是先拆出：

- 反应物列表
- 产物列表
- 每条反应里 reactant / product 的最大 slot 数

原仓库里还有一个容易忽略的小细节：会先把 `~` 替换成 `-`，避免某些配位结构在 RDKit 解析时出现不一致行为。

### 源码对应

- 文件：`data/get_data.py`
- 关键逻辑：`rsmi = rsmi_list[i].replace('~', '-')`、`split('>>')`、`split('.')`

### 下面这个代码单元会做什么？

它会把 reaction SMILES 拆成 reactant 列表和 product 列表，并统计每条反应的 reactant / product 数量。这一步的输出会直接决定后面 slot packing 的形状。



In [ ]:
# ====== Step 2: 解析并标准化 reaction SMILES ======
def normalize_reaction_smiles(rsmi: str) -> str:
    return rsmi.replace('~', '-')


def split_reaction_smiles(rsmi: str) -> tuple[list[str], list[str]]:
    reactants_smi, products_smi = normalize_reaction_smiles(rsmi).split('>>')
    reactants = [s for s in reactants_smi.split('.') if s]
    products = [s for s in products_smi.split('.') if s]
    return reactants, products


parsed_rows = []
for row in demo_df.to_dict('records'):
    reactants, products = split_reaction_smiles(row['reaction_smiles'])
    parsed_rows.append(
        {
            'sample_id': row['sample_id'],
            'yield': float(row['yield']),
            'n_reactants': len(reactants),
            'n_products': len(products),
            'reactants': reactants,
            'products': products,
        }
    )

parsed_df = pd.DataFrame(parsed_rows)
display(parsed_df[['sample_id', 'yield', 'n_reactants', 'n_products']])
print('第 1 条反应的 reactants:')
for idx, smi in enumerate(parsed_df.loc[0, 'reactants']):
    print(f'  reactant[{idx}] = {smi}')
print('第 1 条反应的 products:')
for idx, smi in enumerate(parsed_df.loc[0, 'products']):
    print(f'  product[{idx}] = {smi}')

## 3. 重现 `get_data.py` 的特征工程

### 核心思想

YieldMPNN 并不使用 learned tokenizer，也不直接使用分子图的原始整数属性；它采用的是一套**手工定义的布尔特征**：

- 节点特征（43 维，Buchwald-Hartwig 数据集）
  - 原子类型、形式电荷、度数、杂化、氢数、价态
  - 氢键供体/受体
  - 芳香性、是否在环上、环大小、手性
- 边特征（8 维）
  - 键类型、环属性、共轭属性、双键立体化学

### 源码对应

- 文件：`data/get_data.py`
- 关键内部函数：`_DA()`、`_chirality()`、`_stereochemistry()`、`add_mol()`

### 教学版与原版差异

教学版把这些逻辑拆成更小的函数，并额外保留了特征名字，方便观察每一维到底代表什么。

### 下面这个代码单元会做什么？

它会把 `get_data.py` 里分散的手工特征逻辑重新整理成小函数，并保留特征名。对单个原子来说，可以把节点特征写成：

$$
\mathbf{x}_{\mathrm{atom}} = [
\mathrm{atom\ type};\ \mathrm{charge};\ \mathrm{degree};\ \mathrm{hybridization};\ \mathrm{hydrogen};\ \mathrm{valence};\ \mathrm{donor/acceptor};\ \mathrm{ring/aromatic};\ \mathrm{ring\ size};\ \mathrm{chirality}
 ]
$$

下面这格代码的目标就是把这组布尔特征真正编码出来。



In [ ]:
# ====== Step 3: 定义 YieldMPNN 的节点/边特征 ======
ATOM_LIST = ['C', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Br', 'Pd', 'I']
CHARGE_LIST = [1, 2, -1, 0]
DEGREE_LIST = [1, 2, 3, 4, 0]
VALENCE_LIST = [1, 2, 3, 4, 5, 6, 0]
HYBRIDIZATION_LIST = ['SP', 'SP2', 'SP3', 'SP3D', 'SP3D2', 'S']
HYDROGEN_LIST = [1, 2, 3, 0]
RINGSIZE_LIST = [3, 4, 5, 6, 7, 8]
BOND_LIST = ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']


def one_hot_without_last(value, candidates):
    return np.eye(len(candidates), dtype=bool)[candidates.index(value)][:-1]


def donor_acceptor_atom_ids(mol):
    donors, acceptors = [], []
    for feat in chem_feature_factory.GetFeaturesForMol(mol):
        if feat.GetFamily() == 'Donor':
            donors.append(feat.GetAtomIds()[0])
        if feat.GetFamily() == 'Acceptor':
            acceptors.append(feat.GetAtomIds()[0])
    return set(donors), set(acceptors)


def atom_chirality_bits(atom):
    if atom.HasProp('Chirality'):
        return np.array([
            atom.GetProp('Chirality') == 'Tet_CW',
            atom.GetProp('Chirality') == 'Tet_CCW',
        ], dtype=bool)
    return np.array([False, False], dtype=bool)


def bond_stereo_bits(bond):
    if bond.HasProp('Stereochemistry'):
        return np.array([
            bond.GetProp('Stereochemistry') == 'Bond_Cis',
            bond.GetProp('Stereochemistry') == 'Bond_Trans',
        ], dtype=bool)
    return np.array([False, False], dtype=bool)


def annotate_stereo(mol):
    for element in Chem.FindPotentialStereo(mol):
        if str(element.type) == 'Atom_Tetrahedral' and str(element.specified) == 'Specified':
            mol.GetAtomWithIdx(element.centeredOn).SetProp('Chirality', str(element.descriptor))
        elif str(element.type) == 'Bond_Double' and str(element.specified) == 'Specified':
            mol.GetBondWithIdx(element.centeredOn).SetProp('Stereochemistry', str(element.descriptor))
    return mol


def feature_names():
    node_names = []
    node_names += [f'atom={x}' for x in ATOM_LIST]
    node_names += [f'formal_charge={x}' for x in CHARGE_LIST[:-1]]
    node_names += [f'degree={x}' for x in DEGREE_LIST[:-1]]
    node_names += [f'hybridization={x}' for x in HYBRIDIZATION_LIST[:-1]]
    node_names += [f'total_h={x}' for x in HYDROGEN_LIST[:-1]]
    node_names += [f'total_valence={x}' for x in VALENCE_LIST[:-1]]
    node_names += ['is_donor', 'is_acceptor']
    node_names += ['is_aromatic', 'is_in_ring']
    node_names += [f'in_ring_size={x}' for x in RINGSIZE_LIST]
    node_names += ['chirality_cw', 'chirality_ccw']

    edge_names = []
    edge_names += [f'bond_type={x}' for x in BOND_LIST]
    edge_names += ['bond_in_ring', 'bond_is_conjugated']
    edge_names += ['bond_cis', 'bond_trans']
    return node_names, edge_names


def featurize_molecule(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Cannot parse SMILES: {smiles}')

    mol = annotate_stereo(mol)
    mol = Chem.RemoveHs(mol)
    donors, acceptors = donor_acceptor_atom_ids(mol)

    node_features = []
    for atom_idx, atom in enumerate(mol.GetAtoms()):
        features = np.concatenate([
            np.eye(len(ATOM_LIST), dtype=bool)[ATOM_LIST.index(atom.GetSymbol())],
            one_hot_without_last(atom.GetFormalCharge(), CHARGE_LIST),
            one_hot_without_last(atom.GetDegree(), DEGREE_LIST),
            one_hot_without_last(str(atom.GetHybridization()), HYBRIDIZATION_LIST),
            one_hot_without_last(atom.GetTotalNumHs(includeNeighbors=True), HYDROGEN_LIST),
            one_hot_without_last(atom.GetTotalValence(), VALENCE_LIST),
            np.array([atom_idx in donors, atom_idx in acceptors], dtype=bool),
            np.array([atom.GetIsAromatic(), atom.IsInRing()], dtype=bool),
            np.array([atom.IsInRingSize(size) for size in RINGSIZE_LIST], dtype=bool),
            atom_chirality_bits(atom),
        ])
        node_features.append(features)

    bond_features = []
    src, dst = [], []
    for bond in mol.GetBonds():
        feat = np.concatenate([
            np.eye(len(BOND_LIST), dtype=bool)[BOND_LIST.index(str(bond.GetBondType()))],
            np.array([bond.IsInRing(), bond.GetIsConjugated()], dtype=bool),
            bond_stereo_bits(bond),
        ])
        begin, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_features.extend([feat, feat])
        src.extend([begin, end])
        dst.extend([end, begin])

    return {
        'smiles': smiles,
        'n_node': mol.GetNumAtoms(),
        'n_edge': len(src),
        'node_attr': np.asarray(node_features, dtype=bool),
        'edge_attr': np.asarray(bond_features, dtype=bool).reshape(len(src), -1) if src else np.empty((0, 8), dtype=bool),
        'src': np.asarray(src, dtype=int),
        'dst': np.asarray(dst, dtype=int),
    }


node_feature_names, edge_feature_names = feature_names()
print('node feature dim =', len(node_feature_names))
print('edge feature dim =', len(edge_feature_names))

## 4. 单分子转图示例

### 输出结果说明

这里我们先只看一个 reactant，验证三个关键信息：

1. 原子数和边数是否合理
2. `node_attr` 是否确实是 43 维布尔特征
3. `edge_attr` 是否确实是 8 维布尔特征

这样读者就能把“SMILES -> 图”这个最核心的步骤看清楚，再进入批量压缩逻辑。

### 下面这个代码单元会做什么？

它会拿一个具体 reactant 做单分子级检查，验证 `node_attr` 是否是 43 维、`edge_attr` 是否是 8 维，并把前几行特征打印出来给读者观察。



In [ ]:
# ====== Step 4: 观察单个 reactant 的图特征 ======
example_smiles = parsed_df.loc[0, 'reactants'][0]
example_graph = featurize_molecule(example_smiles)

print('example_smiles =', example_smiles)
print('n_node =', example_graph['n_node'])
print('n_edge =', example_graph['n_edge'])
print('node_attr shape =', example_graph['node_attr'].shape)
print('edge_attr shape =', example_graph['edge_attr'].shape)

atom_feature_preview = pd.DataFrame(
    example_graph['node_attr'][: min(5, example_graph['n_node'])].astype(int),
    columns=node_feature_names,
)
edge_feature_preview = pd.DataFrame(
    example_graph['edge_attr'][: min(6, example_graph['n_edge'])].astype(int),
    columns=edge_feature_names,
)

print('前几个原子的布尔特征预览：')
display(atom_feature_preview.iloc[:, :18])
print('前几条有向边的布尔特征预览：')
display(edge_feature_preview)

## 5. 多反应压缩为仓库风格数据结构

### 为什么这里要引入 slot？

同一个数据集中的不同反应，reactant 数量并不一致。`reaction_yield_nn` 的做法不是把所有分子拼成一个超大图，而是：

- 先统计整个 split 的 `rmol_max_cnt` 和 `pmol_max_cnt`
- 为每个 reactant slot / product slot 各建一个“压缩包”
- 某条反应如果某个 slot 没有分子，就填一个 dummy graph

这样做的好处是：

- 结构简单
- 读取时可以快速按 slot 恢复图
- 顶层模型可以对每个 reactant slot 独立编码，再求和聚合

### 源码对应

- 文件：`data/get_data.py`
- 关键逻辑：`rmol_max_cnt`、`pmol_max_cnt`、`add_dummy()`、`dict_list_to_numpy()`

### 下面这个代码单元会做什么？

它会把多条反应压缩成仓库风格的 slot-based 数组结构，并把教学小样本保存成一个小型 `.npz` 文件。可以把这个压缩过程理解为：

$$
	ext{reaction batch} 
ightarrow \{	ext{reactant slot}_1, \ldots, 	ext{reactant slot}_R, 	ext{product slot}_1, \ldots\}
$$

其中每个 slot 自己维护 `n_node / n_edge / node_attr / edge_attr / src / dst`。



In [ ]:
# ====== Step 5: 把多条反应压缩为仓库风格的 slot 数据 ======
def mol_pack_template():
    return {'n_node': [], 'n_edge': [], 'node_attr': [], 'edge_attr': [], 'src': [], 'dst': []}


def add_dummy(pack):
    pack['n_node'].append(0)
    pack['n_edge'].append(0)
    pack['node_attr'].append(np.empty((0, len(node_feature_names)), dtype=bool))
    return pack


def add_molecule(pack, smiles):
    graph_data = featurize_molecule(smiles)
    pack['n_node'].append(graph_data['n_node'])
    pack['n_edge'].append(graph_data['n_edge'])
    pack['node_attr'].append(graph_data['node_attr'])
    if graph_data['n_edge'] > 0:
        pack['edge_attr'].append(graph_data['edge_attr'])
        pack['src'].append(graph_data['src'])
        pack['dst'].append(graph_data['dst'])
    return pack


def finalize_pack(pack):
    pack['n_node'] = np.asarray(pack['n_node'], dtype=int)
    pack['n_edge'] = np.asarray(pack['n_edge'], dtype=int)
    pack['node_attr'] = np.vstack(pack['node_attr']).astype(bool)
    if pack['n_edge'].sum() > 0:
        pack['edge_attr'] = np.vstack(pack['edge_attr']).astype(bool)
        pack['src'] = np.hstack(pack['src']).astype(int)
        pack['dst'] = np.hstack(pack['dst']).astype(int)
    else:
        pack['edge_attr'] = np.empty((0, len(edge_feature_names)), dtype=bool)
        pack['src'] = np.empty(0, dtype=int)
        pack['dst'] = np.empty(0, dtype=int)
    return pack


def pack_reaction_batch(df: pd.DataFrame):
    reactant_lists = []
    product_lists = []
    for rsmi in df['reaction_smiles']:
        reactants, products = split_reaction_smiles(rsmi)
        reactant_lists.append(reactants)
        product_lists.append(products)

    rmol_max_cnt = max(len(items) for items in reactant_lists)
    pmol_max_cnt = max(len(items) for items in product_lists)

    rmol_packs = [mol_pack_template() for _ in range(rmol_max_cnt)]
    pmol_packs = [mol_pack_template() for _ in range(pmol_max_cnt)]
    reaction_dict = {'yld': df['yield'].to_numpy(dtype=float), 'rsmi': df['reaction_smiles'].to_numpy()}

    for reactants, products in zip(reactant_lists, product_lists):
        reactants = reactants + [''] * (rmol_max_cnt - len(reactants))
        products = products + [''] * (pmol_max_cnt - len(products))

        for slot_idx, smiles in enumerate(reactants):
            rmol_packs[slot_idx] = add_dummy(rmol_packs[slot_idx]) if smiles == '' else add_molecule(rmol_packs[slot_idx], smiles)
        for slot_idx, smiles in enumerate(products):
            pmol_packs[slot_idx] = add_dummy(pmol_packs[slot_idx]) if smiles == '' else add_molecule(pmol_packs[slot_idx], smiles)

    rmol_packs = [finalize_pack(pack) for pack in rmol_packs]
    pmol_packs = [finalize_pack(pack) for pack in pmol_packs]
    return {'rmol': rmol_packs, 'pmol': pmol_packs, 'reaction': reaction_dict}


packed_demo = pack_reaction_batch(demo_df)
DEMO_PACKED_NPZ = TUTORIAL_DIR / 'yieldmpnn_demo_dataset.npz'
np.savez_compressed(DEMO_PACKED_NPZ, rmol=packed_demo['rmol'], pmol=packed_demo['pmol'], reaction=[packed_demo['reaction']])

print('rmol_max_cnt =', len(packed_demo['rmol']))
print('pmol_max_cnt =', len(packed_demo['pmol']))
for slot_idx, pack in enumerate(packed_demo['rmol']):
    print(f"reactant slot {slot_idx}: n_node shape={pack['n_node'].shape}, node_attr shape={pack['node_attr'].shape}, edge_attr shape={pack['edge_attr'].shape}")
for slot_idx, pack in enumerate(packed_demo['pmol']):
    print(f"product slot {slot_idx}: n_node shape={pack['n_node'].shape}, node_attr shape={pack['node_attr'].shape}, edge_attr shape={pack['edge_attr'].shape}")
print('saved mini dataset to:', DEMO_PACKED_NPZ)

## 6. 恢复为 DGL 图对象

### 为什么要做这一步？

到这里我们已经重现了 `get_data.py` 的输出格式，但模型真正吃到的并不是压缩数组，而是 `dataset.py` 在 `__getitem__` 里恢复出来的 DGL 图对象。

所以这一节要回答的是：

- slot pack 如何切回单条反应
- `node_attr / edge_attr / src / dst` 如何重新拼成图
- 为什么 `util.py` 最后可以直接对这些图做 `dgl.batch()`

### 下面这个代码单元会做什么？

它会把上一步压缩后的 slot 数据重新切回单条样本，并恢复成 DGL 图对象。教学目的就是把“压缩存储格式”和“模型实际输入格式”明确打通。



In [ ]:
# ====== Step 6: 从压缩 slot 恢复单条反应的 DGL 图 ======
def cumulative_offsets(lengths: np.ndarray) -> np.ndarray:
    return np.concatenate([[0], np.cumsum(lengths)])


def packed_slot_to_graph(slot_pack: dict, idx: int):
    node_offsets = cumulative_offsets(slot_pack['n_node'])
    edge_offsets = cumulative_offsets(slot_pack['n_edge'])

    start_n, end_n = node_offsets[idx], node_offsets[idx + 1]
    start_e, end_e = edge_offsets[idx], edge_offsets[idx + 1]

    g = dgl.graph(
        (
            slot_pack['src'][start_e:end_e],
            slot_pack['dst'][start_e:end_e],
        ),
        num_nodes=int(slot_pack['n_node'][idx]),
    )
    g.ndata['attr'] = torch.from_numpy(slot_pack['node_attr'][start_n:end_n]).float()
    g.edata['edge_attr'] = torch.from_numpy(slot_pack['edge_attr'][start_e:end_e]).float()
    return g


sample_idx = 0
reactant_graphs = [packed_slot_to_graph(pack, sample_idx) for pack in packed_demo['rmol']]
product_graphs = [packed_slot_to_graph(pack, sample_idx) for pack in packed_demo['pmol']]

for slot_idx, graph in enumerate(reactant_graphs):
    print(f'reactant slot {slot_idx}: nodes={graph.num_nodes()}, edges={graph.num_edges()}, node_attr={tuple(graph.ndata["attr"].shape)}, edge_attr={tuple(graph.edata["edge_attr"].shape)}')
for slot_idx, graph in enumerate(product_graphs):
    print(f'product slot {slot_idx}: nodes={graph.num_nodes()}, edges={graph.num_edges()}, node_attr={tuple(graph.ndata["attr"].shape)}, edge_attr={tuple(graph.edata["edge_attr"].shape)}')

batched_reactants = [dgl.batch([packed_slot_to_graph(pack, i) for i in range(len(demo_df))]) for pack in packed_demo['rmol']]
batched_products = [dgl.batch([packed_slot_to_graph(pack, i) for i in range(len(demo_df))]) for pack in packed_demo['pmol']]

print('\n批处理后：')
print('reactant slot 0 batched node_attr shape =', tuple(batched_reactants[0].ndata['attr'].shape))
print('product slot 0 batched node_attr shape =', tuple(batched_products[0].ndata['attr'].shape))
print('labels shape =', packed_demo['reaction']['yld'].shape)

## 7. 与原仓库处理结果对照

### 为什么要对照原仓库？

教学版代码的目标是“更容易理解”，但不能偏离原始工程的数据约定。最直接的检查方式就是：

- 看维度是否一致
- 看 `rmol_max_cnt / pmol_max_cnt` 是否一致
- 看 `dataset.py` 恢复出的单条样本形状是否与我们手写版本兼容

### 下面这个代码单元会做什么？

它会直接调用原仓库 `dataset.py` 做对照检查，确认教学版在 `rmol_max_cnt / pmol_max_cnt / node_dim / edge_dim` 上与原始工程保持一致。



In [ ]:
# ====== Step 7: 与原仓库的 dataset.py / dataset_1_0.npz 对照 ======
sys.path.insert(0, str(SOURCE_REPO))
from dataset import GraphDataset

with pushd(SOURCE_REPO):
    repo_dataset = GraphDataset(data_id=1, split_id=0)
    repo_sample = repo_dataset[0]

print('原仓库 rmol_max_cnt =', repo_dataset.rmol_max_cnt)
print('原仓库 pmol_max_cnt =', repo_dataset.pmol_max_cnt)
print('原仓库 node_dim =', repo_dataset.rmol_node_attr[0].shape[1])
print('原仓库 edge_dim =', repo_dataset.rmol_edge_attr[0].shape[1])
print('教学版 demo rmol_max_cnt =', len(packed_demo['rmol']))
print('教学版 demo pmol_max_cnt =', len(packed_demo['pmol']))
print('教学版 node_dim =', packed_demo['rmol'][0]['node_attr'].shape[1])
print('教学版 edge_dim =', packed_demo['rmol'][0]['edge_attr'].shape[1])
print()
print('原仓库第 1 条样本 tuple 长度 =', len(repo_sample))
print('原仓库第 1 条标签 =', repo_sample[-1])
print('原仓库第 1 个 reactant graph 的 node_attr shape =', tuple(repo_sample[0].ndata['attr'].shape))
print('原仓库第 1 个 product graph 的 node_attr shape =', tuple(repo_sample[-2].ndata['attr'].shape))

## 8. 总结

本 notebook 依次完成了：

1. 从原始 reaction SMILES 读取教学小样本。
2. 重现 `data/get_data.py` 中的分子特征构造逻辑。
3. 演示如何把多条反应压缩成 slot-based 的仓库风格数组结构。
4. 再把这些压缩结构恢复成 DGL 图对象，对接后续模型输入。

### 关键概念回顾

- `rmol_max_cnt / pmol_max_cnt`：整个 split 的最大 reactant / product slot 数
- `node_attr / edge_attr`：YieldMPNN 的布尔图特征
- `dataset.py`：压缩数组到 DGL 图对象的恢复层

### 下一步

- 下一份 notebook：`3_模型展示.ipynb`
- 重点内容：`MPNN`、`reactionMPNN`、MC dropout 和不确定性分解